In [ ]:
import sys
import os
depth_v2 = os.path.abspath("depth_anything_v2\metric_depth")
sys.path.append(depth_v2)

In [ ]:
import cv2
import serial
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from matplotlib import cm
import matplotlib.animation as animation
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import ultralytics
from ultralytics import YOLO
import torch
from IPython.display import clear_output
import time


In [ ]:
PORT = 'COM6' 
BAUD_RATE = 9600

In [ ]:
try:
    ser = serial.Serial(PORT, BAUD_RATE, timeout=1)
    print(f"--- Connected to {PORT} at {BAUD_RATE} baud ---")
    print("Type characters to send to STM32 (Ctrl+C to exit)\n")
except:
    print("not connected_retry")

In [ ]:
from depth_anything_v2.dpt import DepthAnythingV2

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

model_configs = {
    'vits': {'encoder': 'vits', 'features': 64, 'out_channels': [48, 96, 192, 384]},
    'vitb': {'encoder': 'vitb', 'features': 128, 'out_channels': [96, 192, 384, 768]},
    'vitl': {'encoder': 'vitl', 'features': 256, 'out_channels': [256, 512, 1024, 1024]},
    'vitg': {'encoder': 'vitg', 'features': 384, 'out_channels': [1536, 1536, 1536, 1536]}
}

encoder = 'vits' # Using 'vits' because depth_anything_v2_vits.pth was downloaded
dataset = 'vkitti' # 'hypersim' for indoor model, 'vkitti' for outdoor model
max_depth = 20 # 20 for indoor model, 80 for outdoor model

model = DepthAnythingV2(**{**model_configs[encoder], 'max_depth': max_depth})
model.load_state_dict(torch.load(f'checkpoints/depth_anything_v2_metric_hypersim_vits.pth', map_location='cpu'))
model.eval()
model = model.to(DEVICE).eval()

In [ ]:
def classify_danger_zones(pin_heights):
    """
    Returns a danger mask:  2=near, 1=mid, 0=far
    Used for haptic/audio alert prioritisation.
    """
    mask = np.zeros_like(pin_heights, dtype=np.uint8)
    mask[pin_heights > (1 - cfg.DANGER_NEAR)] = 2
    mask[(pin_heights > (1 - cfg.DANGER_MID)) & (mask == 0)] = 1
    return mask

In [ ]:
class TactileConfig:
    DEVICE_WIDTH_MM = 80
    DEVICE_HEIGHT_MM = 140

    PIN_COLS = 5
    PIN_ROWS = 4
    TOTAL_PINS = PIN_COLS * PIN_ROWS
    PIN_MIN_HEIGHT = 0.0
    PIN_MAX_HEIGHT = 8.0
    PIN_DIAMETER_MM = 2.5

    DANGER_NEAR    = 0.3
    DANGER_MID     = 0.6
    DANGER_FAR     = 1.0

    TEMPORAL_ALPHA = 0.4     # IIR filter: 0=no update, 1=instant
cfg = TactileConfig()


In [ ]:
# ============================================================
# CORE: DEPTH → PIN HEIGHT MAPPING
# ============================================================
def depth_to_pin_heights(depth_map, prev_heights=None):
    """
    Converts a full-resolution depth map into a PIN_ROWS x PIN_COLS
    matrix of normalised pin heights in [0, 1].

    Closer objects  → taller pins  (inverse depth mapping)
    """
    h, w = depth_map.shape

    # Resize depth map to pin grid resolution
    grid = cv2.resize(depth_map, (cfg.PIN_COLS, cfg.PIN_ROWS),
                      interpolation=cv2.INTER_AREA)

    # Normalise to [0, 1]
    d_min, d_max = grid.min(), grid.max()
    if d_max - d_min < 1e-6:
        normalised = np.zeros_like(grid)
    else:
        normalised = (grid - d_min) / (d_max - d_min)

    # INVERT: closer = higher pin
    pin_heights = 1.0 - normalised

    # Temporal smoothing (prevents jitter between frames)
    if prev_heights is not None:
        pin_heights = cfg.TEMPORAL_ALPHA * pin_heights + \
                      (1 - cfg.TEMPORAL_ALPHA) * prev_heights

    return pin_heights   # shape: (PIN_ROWS, PIN_COLS)

In [ ]:
def draw_tactile_grid_2d(ax, pin_heights, danger_mask, title="Tactile Pin Grid"):
    """
    Draws the pin grid as circles whose SIZE represents pin height.
    Color encodes danger zone.
    """
    ax.set_facecolor('#1a1a2e')
    ax.set_xlim(-0.5, cfg.PIN_COLS - 0.5)
    ax.set_ylim(-0.5, cfg.PIN_ROWS - 0.5)
    ax.set_aspect('equal')
    ax.set_title(title, color='white', fontsize=11, fontweight='bold')
    ax.tick_params(colors='gray')
    for spine in ax.spines.values():
        spine.set_edgecolor('#444')

    # Color palette per danger level
    color_map = {0: '#4a9eff', 1: '#ffaa00', 2: '#ff4444'}

    rows, cols = pin_heights.shape
    for r in range(rows):
        for c in range(cols):
            h   = pin_heights[r, c]          # 0..1
            lvl = danger_mask[r, c]
            color = color_map[lvl]

            # Max radius so pins just touch at full extension
            max_r = 0.42
            radius = max_r * h + 0.04        # small base always visible

            circle = plt.Circle((c, rows - 1 - r), radius,
                                 color=color, alpha=0.85)
            ax.add_patch(circle)

    # Legend
    patches = [
        mpatches.Patch(color='#4a9eff', label='Clear'),
        mpatches.Patch(color='#ffaa00', label='Caution'),
        mpatches.Patch(color='#ff4444', label='Danger'),
    ]
    ax.legend(handles=patches, loc='upper right',
              facecolor='#2a2a3e', labelcolor='white', fontsize=8)

In [ ]:
# ============================================================
# VISUALISATION: 3D PIN ARRAY (isometric bar chart)
# ============================================================
def draw_tactile_grid_3d(ax, pin_heights, danger_mask):
    """
    Renders pins as 3-D bars rising from a base plate.
    Height = physical pin extension (0..PIN_MAX_HEIGHT mm).
    """
    ax.set_facecolor('#0d0d1a')
    ax.set_title("3D Pin Array (Side View)", color='white',
                 fontsize=11, fontweight='bold')

    rows, cols = pin_heights.shape
    xpos = np.arange(cols)
    ypos = np.arange(rows)
    xpos, ypos = np.meshgrid(xpos, ypos)
    xpos = xpos.flatten()
    ypos = ypos.flatten()
    zpos = np.zeros_like(xpos)

    heights_mm = pin_heights.flatten() * cfg.PIN_MAX_HEIGHT

    # Colour by danger
    color_map = {0: '#4a9eff', 1: '#ffaa00', 2: '#ff4444'}
    colors = [color_map[danger_mask.flatten()[i]] for i in range(len(heights_mm))]

    dx = dy = 0.7
    ax.bar3d(xpos, ypos, zpos, dx, dy, heights_mm,
             color=colors, alpha=0.8, shade=True)

    ax.set_xlabel('X', color='gray', fontsize=8)
    ax.set_ylabel('Y', color='gray', fontsize=8)
    ax.set_zlabel('Height (mm)', color='gray', fontsize=8)
    ax.set_zlim(0, cfg.PIN_MAX_HEIGHT)
    ax.tick_params(colors='gray', labelsize=7)
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False

In [ ]:
def overlay_yolo_on_pins(pin_heights, boxes, frame_shape):
    """
    For each YOLO bounding box, boost pin heights in that region
    so detected objects are prominently raised on the tactile display.
    """
    fh, fw = frame_shape[:2]
    highlighted = pin_heights.copy()

    for box in boxes:
        x1, y1, x2, y2 = box
        # Map pixel coords → pin grid coords
        pc1 = int(x1 / fw * cfg.PIN_COLS)
        pc2 = int(x2 / fw * cfg.PIN_COLS)
        pr1 = int(y1 / fh * cfg.PIN_ROWS)
        pr2 = int(y2 / fh * cfg.PIN_ROWS)
        # Raise pins inside the box to at least 0.8
        highlighted[pr1:pr2, pc1:pc2] = np.maximum(
            highlighted[pr1:pr2, pc1:pc2], 0.8)

    return highlighted

In [ ]:
def classify_danger_zones(pin_heights):
    """
    Returns a danger mask:  2=near, 1=mid, 0=far
    Used for haptic/audio alert prioritisation.
    """
    mask = np.zeros_like(pin_heights, dtype=np.uint8)
    mask[pin_heights > (1 - cfg.DANGER_NEAR)] = 2
    mask[(pin_heights > (1 - cfg.DANGER_MID)) & (mask == 0)] = 1
    return mask

In [ ]:
# ============================================================
# SINGLE-FRAME FULL PIPELINE
# ============================================================
def process_frame_tactile(frame, model, model_yolo,
                           prev_pin_heights=None,
                           view_angle=(30, -60)):
    """
    Full pipeline for one frame.
    Returns: (fig, pin_heights) so pin_heights feeds into next frame.
    """
    focal_length_x = focal_length_y = 470.4
    h, w = frame.shape[:2]

    # --- Depth estimation ---
    color_image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    pred  = model.infer_image(frame, h)
    depth = cv2.resize(pred, (w, h))

    # Depth colormap
    depth_norm = cv2.normalize(depth, None, 0, 255,
                               cv2.NORM_MINMAX, dtype=cv2.CV_8U)
    depth_color = cv2.applyColorMap(depth_norm, cv2.COLORMAP_MAGMA)
    depth_rgb   = cv2.cvtColor(depth_color, cv2.COLOR_BGR2RGB)

    # --- YOLO detection ---
    frame_rgb = color_image.copy()
    results   = model_yolo(frame)
    all_boxes = []
    for result in results:
        boxes = result.boxes.xyxy.cpu().numpy()
        for box in boxes:
            x1, y1, x2, y2 = map(int, box)
            cv2.rectangle(frame_rgb, (x1, y1), (x2, y2), (0, 255, 80), 2)
            all_boxes.append((x1, y1, x2, y2))

    # --- Pin height computation ---
    pin_heights  = depth_to_pin_heights(depth, prev_pin_heights)
    pin_heights  = overlay_yolo_on_pins(pin_heights, all_boxes, frame.shape)
    danger_mask  = classify_danger_zones(pin_heights)

    # ---- FIGURE LAYOUT ----
    fig = plt.figure(figsize=(20, 10), facecolor='#0d0d1a')
    gs  = fig.add_gridspec(2, 4, hspace=0.35, wspace=0.3)

    # Panel 1: Original + YOLO
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.imshow(frame_rgb)
    ax1.set_title("Camera + YOLO", color='white', fontweight='bold')
    ax1.axis('off')

    # Panel 2: Depth map
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.imshow(depth_rgb)
    ax2.set_title("Depth Map (Magma)", color='white', fontweight='bold')
    ax2.axis('off')

    # Panel 3: 2D tactile grid
    ax3 = fig.add_subplot(gs[0, 2])
    draw_tactile_grid_2d(ax3, pin_heights, danger_mask,
                         title="Tactile Grid (Top View)")

    # Panel 4: Danger heatmap
    ax4 = fig.add_subplot(gs[0, 3])
    ax4.imshow(danger_mask, cmap='RdYlGn_r', vmin=0, vmax=2,
               interpolation='nearest', aspect='auto')
    ax4.set_title("Danger Zone Map", color='white', fontweight='bold')
    ax4.axis('off')
    cb = plt.colorbar(ax4.images[0], ax=ax4, fraction=0.046)
    cb.set_ticks([0, 1, 2])
    cb.set_ticklabels(['Clear', 'Caution', 'Danger'])
    cb.ax.yaxis.set_tick_params(color='white')
    plt.setp(cb.ax.yaxis.get_ticklabels(), color='white')

    # Panel 5: 3D pin array (spans bottom half)
    ax5 = fig.add_subplot(gs[1, :3], projection='3d')
    ax5.view_init(*view_angle)
    draw_tactile_grid_3d(ax5, pin_heights, danger_mask)


    plt.suptitle(
        f"Tactile Display Simulator  |  "
        f"{cfg.PIN_ROWS}×{cfg.PIN_COLS} pins  |  "
        f"Max travel {cfg.PIN_MAX_HEIGHT} mm",
        color='white', fontsize=13, fontweight='bold', y=1.01)

    return fig, pin_heights

In [ ]:
model_yolo = YOLO('yolov8n.pt')  # ultralytics YOLOv8 nano (fast, less accurate)


In [ ]:
# ============================================================
# VIDEO PROCESSING LOOP
# ============================================================
def run_tactile_video(video_path, model, model_yolo,
                      max_frames=None, skip=1,
                      save_output=True,
                      output_path="tactile_output.mp4"):
    """
    Processes every `skip`-th frame of a video.
    Displays inline in Colab and optionally writes an output video.
    """
    video = cv2.VideoCapture(video_path)
    fps   = video.get(cv2.CAP_PROP_FPS)
    iw    = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
    ih    = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))

    out_writer = None   # lazy init after first fig render

    prev_heights = None
    frame_idx    = 0
    written      = 0
    print(f"📹 Video: {iw}×{ih} @ {fps:.1f} fps")
    print(f"📌 Pin grid: {cfg.PIN_ROWS}×{cfg.PIN_COLS} = {cfg.TOTAL_PINS} pins")
    print(f"🔁 Processing every {skip} frame(s)…\n")

    while video.isOpened():
        ret, frame = video.read()
        frame=cv2.resize(frame,(100,100))
        if not ret:
            break
        if frame_idx % skip != 0:
            frame_idx += 1
            continue
        if max_frames and written >= max_frames:
            break

        fig, prev_heights = process_frame_tactile(
            frame, model, model_yolo, prev_heights)
        
        #the display controller
        prev = prev_heights*20
        row,col=prev.shape
        print(row,col)
        for r in range(row):
            for c in range(col):
                message_bytes = bytearray(b"A09909909909911111")
                h = prev[r, c]
                h_bytes = str(int(h)).zfill(3).encode('utf-8')
                message_bytes[r*4+1 : r*4+4] = h_bytes
                message_bytes[13 + c] = ord('0')
                ser.write(message_bytes)
                #time.sleep(0.001)
        #display controller ends
        fig.canvas.draw()
        buf = np.array(fig.canvas.renderer.buffer_rgba())
        buf = buf[:, :, :3] # drop alpha channel → RGB

        if save_output:
            if out_writer is None:
                oh, ow = buf.shape[:2]
                fourcc = cv2.VideoWriter_fourcc(*'mp4v')
                out_writer = cv2.VideoWriter(
                    output_path, fourcc, fps / skip, (ow, oh))
            out_writer.write(cv2.cvtColor(buf, cv2.COLOR_RGB2BGR))
            
        clear_output(wait=True)
        plt.show()
        plt.close(fig)

        frame_idx += 1
        written   += 1
        print(f"  ✅ Frame {frame_idx}  ({written} rendered)", end='\r')

    video.release()
    if out_writer:
        out_writer.release()
        print(f"\n\n💾 Saved: {output_path}")
    print(f"\n🏁 Done — {written} frames processed.")

In [ ]:
run_tactile_video(
    video_path = "vid.mp4",
    model      = model,        # your depth model
    model_yolo = model_yolo,   # your YOLO model
    skip       = 1,            # process every 1st frame
    max_frames = None,           # set None for full video
)